## Defensive Bowling

I noticed a lot of bowlers trying to hide the ball from the batsmen this season. In doing so many of them bowled extremely wide lines consistently, and in doing so conceded lots of runs in wide deliveries. So is this strategy actually effective?

In [25]:
import pandas as pd

In [26]:
cleaned_all_matches_df = pd.read_csv('cleaned_all_matches.csv')

/var/folders/qh/dg_1llp91pjb4yfbj6kj3cc80000gn/T/ipykernel_89938/2952668246.py:1: DtypeWarning: Columns (0: umpires_call, 1: injury_subbed_in, 2: injury_subbed_out, 3: is_non_boundary) have mixed types. Specify dtype option on import or set low_memory=False.
  cleaned_all_matches_df = pd.read_csv('cleaned_all_matches.csv')


In [ ]:
# ensure total_runs numeric
cleaned_all_matches_df["total_runs"] = pd.to_numeric(
    cleaned_all_matches_df.get("total_runs", 0), errors="coerce"
).fillna(0)

# pick over identifier columns (fallback if team not present)
cand = ["source_file", "team", "over"]
over_id_cols = [c for c in cand if c in cleaned_all_matches_df.columns]
if "over" not in over_id_cols:
    raise KeyError("`over` column not found in cleaned_all_matches_df")

# per-over total runs
over_total = (
    cleaned_all_matches_df
    .groupby(over_id_cols, as_index=False)["total_runs"]
    .sum()
    .rename(columns={"total_runs": "over_total_runs"})
)

# one bowler record per over
bowler_over = (
    cleaned_all_matches_df[["bowler", *over_id_cols]]
    .dropna(subset=["bowler"])
    .drop_duplicates()
)

# attach over totals and compute per-bowler average
bowler_over_runs = bowler_over.merge(over_total, on=over_id_cols, how="left")

bowler_economy = (
    bowler_over_runs
    .groupby("bowler", as_index=False)["over_total_runs"]
    .mean()
    .rename(columns={"over_total_runs": "economy"})
)

# round and sort
bowler_economy["economy"] = bowler_economy["economy"].round(2)
bowler_economy = bowler_economy.sort_values("economy", ascending=False).reset_index(drop=True)

# result
display(bowler_economy)

,bowler,economy
0,TH David,18.00
1,M Markande,15.20
2,T Vijay,14.50
3,DA Payne,14.20
4,A Badoni,14.00
...,...,...
119,Harpreet Brar,7.50
120,JO Holder,7.24
121,SP Narine,6.84
122,M Pathirana,6.00


In [28]:
cleaned_all_matches_df["wides"] = pd.to_numeric(cleaned_all_matches_df["wides"], errors="coerce").fillna(0)

# Keep overs separated by match/innings context
group_cols = ["source_file", "team", "over"]

# Count non-zero wides readings per over, in delivery order
over_wides = (
    cleaned_all_matches_df
    .sort_values(group_cols + ["ball_number"])
    .assign(is_wide_reading=lambda df: df["wides"].ne(0).astype(int))
    .groupby(group_cols, as_index=False)["is_wide_reading"]
    .sum()
    .rename(columns={"is_wide_reading": "wide_reading_count"})
)

# Overs with 2 or more non-zero wides readings
overs_with_2plus_wides = over_wides[over_wides["wide_reading_count"] >= 2]

overs_with_2plus_wides

def_overs = overs_with_2plus_wides[group_cols]

filtered_rows = cleaned_all_matches_df.merge(def_overs, on=group_cols, how="inner")
filtered_rows

,team,over,ball_number,powerplays_from,powerplays_to,powerplays_type,striker,bowler,non_striker,batter_runs,...,target_overs,target_runs,current_team_total,source_file,noballs,is_fielder_sub,umpires_call,injury_subbed_in,injury_subbed_out,is_non_boundary
0,Sunrisers Hyderabad,1,1.0,0.1,5.6,mandatory,TM Head,B Kumar,Abhishek Sharma,1,...,NaN,NaN,8,1527674.csv,NaN,NaN,NaN,NaN,NaN,NaN
1,Sunrisers Hyderabad,1,2.0,0.1,5.6,mandatory,Abhishek Sharma,B Kumar,TM Head,1,...,NaN,NaN,9,1527674.csv,NaN,NaN,NaN,NaN,NaN,NaN
2,Sunrisers Hyderabad,1,3.0,0.1,5.6,mandatory,TM Head,B Kumar,Abhishek Sharma,1,...,NaN,NaN,10,1527674.csv,NaN,NaN,NaN,NaN,NaN,NaN
3,Sunrisers Hyderabad,1,4.0,0.1,5.6,mandatory,Abhishek Sharma,B Kumar,TM Head,0,...,NaN,NaN,11,1527674.csv,NaN,NaN,NaN,NaN,NaN,NaN
4,Sunrisers Hyderabad,1,5.0,0.1,5.6,mandatory,Abhishek Sharma,B Kumar,TM Head,0,...,NaN,NaN,11,1527674.csv,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1020,Kolkata Knight Riders,6,4.0,0.1,5.6,mandatory,MK Pandey,M Tiwari,AM Rahane,0,...,20.0,204.0,58,1529313.csv,NaN,NaN,NaN,NaN,NaN,NaN
1021,Kolkata Knight Riders,6,5.0,0.1,5.6,mandatory,MK Pandey,M Tiwari,AM Rahane,0,...,20.0,204.0,58,1529313.csv,NaN,NaN,NaN,NaN,NaN,NaN
1022,Kolkata Knight Riders,6,6.0,0.1,5.6,mandatory,MK Pandey,M Tiwari,AM Rahane,0,...,20.0,204.0,59,1529313.csv,NaN,NaN,NaN,NaN,NaN,NaN
1023,Kolkata Knight Riders,6,7.0,0.1,5.6,mandatory,MK Pandey,M Tiwari,AM Rahane,4,...,20.0,204.0,63,1529313.csv,NaN,NaN,NaN,NaN,NaN,NaN


In [30]:
# keep only the columns needed to identify a unique over
over_id_cols = ["source_file", "team", "over"]

# one record per bowler per over
bowler_overs = (
    filtered_rows[["bowler", *over_id_cols]]
    .dropna(subset=["bowler"])
    .drop_duplicates()
)

# number of overs bowled by each distinct bowler reading
bowler_over_counts = (
    bowler_overs
    .groupby("bowler", as_index=False)
    .size()
    .rename(columns={"size": "overs_bowled"})
    .sort_values(["overs_bowled", "bowler"], ascending=[False, True])
)

# --- per-over total_runs (sum) ---
# ensure total_runs is numeric on the filtered rows
filtered_rows["total_runs"] = pd.to_numeric(filtered_rows.get("total_runs", 0), errors="coerce").fillna(0)

over_total_runs = (
    filtered_rows
    .groupby(over_id_cols, as_index=False)["total_runs"]
    .sum()
    .rename(columns={"total_runs": "over_total_runs"})
)

over_batter_runs = (
    filtered_rows
    .groupby(over_id_cols, as_index=False)["batter_runs"]
    .sum()
    .rename(columns={"batter_runs": "over_batter_runs"})
)

# attach over totals to each bowler-over row and compute per-bowler average
bowler_over_runs = bowler_overs.merge(over_total_runs, on=over_id_cols, how="left")
bowler_over_runs = bowler_over_runs.merge(over_batter_runs, on=over_id_cols, how="left")

bowler_eco = (
    bowler_over_runs
    .groupby("bowler", as_index=False)["over_total_runs"]
    .mean()
    .rename(columns={"over_total_runs": "def_over_economy"})
)

bowler_eco_no_wides = (
    bowler_over_runs
    .groupby("bowler", as_index=False)["over_batter_runs"]
    .mean()
    .rename(columns={"over_batter_runs": "def_over_economy_no_wides"})
)

# round the per-bowler average to 2 decimal places
bowler_eco["def_over_economy"] = bowler_eco["def_over_economy"].round(2)
bowler_eco_no_wides["def_over_economy_no_wides"] = bowler_eco_no_wides["def_over_economy_no_wides"].round(2)

# combine counts and averages for a concise per-bowler view
bowler_stats = (
    bowler_over_counts.merge(bowler_eco, on="bowler", how="left")
    .merge(bowler_eco_no_wides, on="bowler", how="left")
    .fillna({"def_over_economy": 0, "def_over_economy_no_wides": 0})
    .sort_values(["overs_bowled", "def_over_economy"], ascending=[False, False])
)

# dataset-level average over total_runs (across the filtered overs)
dataset_avg_over = over_total_runs["over_total_runs"].mean()

print(f"Dataset average over total_runs: {dataset_avg_over:.3f}")

bowler_stats = bowler_stats.merge(bowler_economy[["bowler", "economy"]], on="bowler", how="left")

bowler_stats["def_over_better"] = bowler_stats["def_over_economy"] < bowler_stats["economy"]
bowler_stats["def_over_no_wides_better"] = bowler_stats["def_over_economy_no_wides"] < bowler_stats["economy"]

display(bowler_stats)

Dataset average over total_runs: 12.290


,bowler,overs_bowled,def_over_economy,def_over_economy_no_wides,economy,def_over_better,def_over_no_wides_better
0,AS Roy,7,8.86,6.29,9.43,True,True
1,Mohammed Shami,6,9.83,7.17,9.25,False,True
2,Arshdeep Singh,5,13.60,9.80,10.23,False,True
3,E Malinga,5,11.80,9.00,9.22,False,True
4,SN Thakur,4,14.50,11.75,12.27,False,True
...,...,...,...,...,...,...,...
57,Vijaykumar Vyshak,1,8.00,6.00,10.74,True,True
58,M Pathirana,1,7.00,5.00,6.00,False,True
59,Avesh Khan,1,5.00,2.00,11.12,True,True
60,SH Johnson,1,5.00,2.00,10.36,True,True


In [31]:
bowler_stats.to_csv('bowler_stats.csv', index=False)

In [32]:
def_over_better_true = bowler_stats["def_over_better"].sum()
def_over_better_pct = bowler_stats["def_over_better"].mean() * 100

def_over_no_wides_better_true = bowler_stats["def_over_no_wides_better"].sum()
def_over_no_wides_better_pct = bowler_stats["def_over_no_wides_better"].mean() * 100

print(f"Percentage of bowlers with better economy in defensive overs: {def_over_better_pct:.2f}% ({def_over_better_true} out of {len(bowler_stats)})")
print(f"Percentage of bowlers with better economy in defensive overs (no wides): {def_over_no_wides_better_pct:.2f}% ({def_over_no_wides_better_true} out of {len(bowler_stats)})")

Percentage of bowlers with better economy in defensive overs: 22.58% (14 out of 62)
Percentage of bowlers with better economy in defensive overs (no wides): 69.35% (43 out of 62)
